# 02 — Graph Construction · Phase 3

**Purpose:** Load the built graph, verify every assertion, visualise the node distribution, edge structure, feature correlations, and geographic split boundaries.

**Pre-condition:** Run `python scripts/phase3_build_graph.py` first. This notebook is for verification and exploration only — it does not rebuild the graph.

**What you confirm here before Phase 4:**
- Graph has the correct number of nodes, features, and edges
- val_mask is not zero (previous project bug)
- No geographic overlap between train/val/test splits
- Target variable y is quantile-transformed (mean≈0, std≈1)
- Feature matrix has no NaN or Inf values
- Node distribution across Greece looks spatially correct

---

In [1]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

import torch
from torch_geometric.data import Data

# Add src/ to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from wildfire_gnn.utils import load_yaml_config, set_seed

config  = load_yaml_config(PROJECT_ROOT / 'configs' / 'gnn_config.yaml')
set_seed(config['training']['seed'])

p           = config['paths']
GRAPH_PATH  = PROJECT_ROOT / p['graph_data']
SPLIT_PATH  = PROJECT_ROOT / p['spatial_split_path']
FEAT_PATH   = PROJECT_ROOT / p['feature_names']
FIG_DIR     = PROJECT_ROOT / p['figures_dir']
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Graph path   : {GRAPH_PATH}')
print(f'Graph exists : {GRAPH_PATH.exists()}')

Project root : d:\wildfire\spatiotemporal_wildfire_gnn
Graph path   : d:\wildfire\spatiotemporal_wildfire_gnn\data\processed\graph_data_enriched.pt
Graph exists : False


# Load graph and build summary

In [ ]:
if not GRAPH_PATH.exists():
    print('ERROR: graph_data_enriched.pt not found.')
    print('Run: python scripts/phase3_build_graph.py')
    raise FileNotFoundError(str(GRAPH_PATH))

graph = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)

print('Graph loaded successfully')
print()
print(f'  num_nodes          : {graph.num_nodes:,}')
print(f'  num_node_features  : {graph.num_node_features}')
print(f'  num_edges          : {graph.num_edges:,}')
print(f'  avg edges per node : {graph.num_edges / graph.num_nodes:.1f}')
print()
print(f'  x shape            : {tuple(graph.x.shape)}')
print(f'  y shape            : {tuple(graph.y.shape)}')
print(f'  pos shape          : {tuple(graph.pos.shape)}')
print(f'  edge_index shape   : {tuple(graph.edge_index.shape)}')
print()
print(f'  train_mask.sum()   : {graph.train_mask.sum():,}')
print(f'  val_mask.sum()     : {graph.val_mask.sum():,}')
print(f'  test_mask.sum()    : {graph.test_mask.sum():,}')
print()
print(f'  y mean             : {float(graph.y.mean()):.4f}  (should be near 0)')
print(f'  y std              : {float(graph.y.std()):.4f}   (should be near 1)')
print(f'  y_raw min          : {float(graph.y_raw.min()):.6f}')
print(f'  y_raw max          : {float(graph.y_raw.max()):.4f}')

# Critical assertion

In [ ]:
print('Running Phase 3 completion assertions...')
print('=' * 55)

failures = []

def check(condition, message, fix=''):
    sym = '✓' if condition else '✗'
    print(f'  {sym}  {message}')
    if not condition:
        failures.append((message, fix))

# Node count
check(graph.num_nodes > 100_000,
      f'num_nodes={graph.num_nodes:,} > 100k',
      'Re-run phase3_build_graph.py with --stride 6')

check(graph.num_nodes < 600_000,
      f'num_nodes={graph.num_nodes:,} < 600k (memory feasible)',
      'Use --stride 6 or higher')

# Feature count
check(graph.num_node_features in (53, 58),
      f'num_features={graph.num_node_features} is 53 (no DEM) or 58 (with DEM)',
      'Check feature_engineering.py output')

# Edge count
check(graph.num_edges > graph.num_nodes * 2,
      f'num_edges={graph.num_edges:,} > 2×nodes (graph is connected)',
      'Check build_pixel_grid_edges() in graph_builder.py')

# Target transformation
y_mean = float(graph.y.mean())
y_std  = float(graph.y.std())
check(abs(y_mean) < 0.5,
      f'y.mean()={y_mean:.4f} near 0 (QuantileTransformer applied)',
      'Transform was not applied or was applied twice')

check(0.5 < y_std < 2.0,
      f'y.std()={y_std:.4f} near 1 (QuantileTransformer applied)',
      'Check TargetTransformer.transform() in target_engineering.py')

# No NaN in features
has_nan = torch.isnan(graph.x).any()
check(not has_nan,
      'Feature matrix x has no NaN values',
      'Check nan_to_num() calls in feature_engineering.py')

# No Inf in features
has_inf = torch.isinf(graph.x).any()
check(not has_inf,
      'Feature matrix x has no Inf values',
      'Check gradient computation in add_spatial_gradients()')

# Split masks non-zero
check(graph.val_mask.sum() > 0,
      f'val_mask.sum()={graph.val_mask.sum():,} > 0 (not zero placeholder)',
      'CRITICAL: val rows range in config produces no nodes — check split.val_rows')

check(graph.test_mask.sum() > 0,
      f'test_mask.sum()={graph.test_mask.sum():,} > 0',
      'Check split.test_rows range in gnn_config.yaml')

# No geographic overlap
tv_overlap = int((graph.train_mask & graph.val_mask).sum())
tt_overlap = int((graph.train_mask & graph.test_mask).sum())
check(tv_overlap == 0,
      f'Train/Val overlap = {tv_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

check(tt_overlap == 0,
      f'Train/Test overlap = {tt_overlap} (must be 0)',
      'CRITICAL geographic leakage — check build_geographic_split()')

# All nodes covered
covered = int(graph.train_mask.sum() + graph.val_mask.sum() + graph.test_mask.sum())
check(covered == graph.num_nodes,
      f'All {graph.num_nodes:,} nodes covered by exactly one split',
      'Check test_rows upper bound in gnn_config.yaml')

# edge_index valid range
check(int(graph.edge_index.max()) < graph.num_nodes,
      f'edge_index max={int(graph.edge_index.max())} < num_nodes={graph.num_nodes}',
      'Edge index contains out-of-range node index')

# dtype checks
check(graph.x.dtype == torch.float32,
      f'x.dtype = {graph.x.dtype} (float32)',
      'Cast x to float32 in graph assembly')

check(graph.edge_index.dtype == torch.long,
      f'edge_index.dtype = {graph.edge_index.dtype} (torch.long/int64)',
      'Cast edge_index to torch.long')

print('=' * 55)
if failures:
    print(f'\n  FAILED: {len(failures)} assertion(s)')
    for msg, fix in failures:
        print(f'  ✗  {msg}')
        if fix:
            print(f'     Fix: {fix}')
    print('\n  Do NOT proceed to Phase 4 until all assertions pass.')
else:
    print(f'\n  ALL {15} ASSERTIONS PASSED — ready for Phase 4')

# Feature names and group breakdown

In [ ]:
if FEAT_PATH.exists():
    with open(FEAT_PATH) as f:
        feature_names = json.load(f)
    print(f'Total features: {len(feature_names)}')
    print()

    groups = [
        ('Base rasters',      [n for n in feature_names if n in ['CFL','FSP_Index','Ignition_Prob','Struct_Exp_Index']]),
        ('DEM terrain',       [n for n in feature_names if n.startswith('dem_')]),
        ('Fuel one-hot',      [n for n in feature_names if n.startswith('fuel_')]),
        ('Interactions',      [n for n in feature_names if n.startswith('interact_')]),
        ('Multi-scale stats', [n for n in feature_names if '_mean_' in n or '_std_' in n]),
        ('Spatial gradients', [n for n in feature_names if '_grad_' in n]),
        ('Node degree',       [n for n in feature_names if n == 'node_degree']),
    ]

    print(f'  {"Group":<22} {"Count":<8} Features')
    print(f'  {"-"*70}')
    for gname, gfeats in groups:
        feat_str = ', '.join(gfeats[:4])
        if len(gfeats) > 4:
            feat_str += f' ... (+{len(gfeats)-4} more)'
        print(f'  {gname:<22} {len(gfeats):<8} {feat_str}')
    total = sum(len(g) for _, g in groups)
    print(f'  {"-"*70}')
    print(f'  {"TOTAL":<22} {total}')
else:
    print('feature_names.json not found — run phase3_build_graph.py first')

# Spatial distribution of subsampled nodes

In [ ]:
# pos contains (original_row, original_col) for each node
pos     = graph.pos.numpy()
rows_np = pos[:, 0].astype(int)
cols_np = pos[:, 1].astype(int)

# Get split boundaries from config
train_end = config['split']['train_rows'][1]
val_end   = config['split']['val_rows'][1]

# Build 2D density map of subsampled nodes
H, W = 7597, 7555
# Downsample for plotting (every 6th pixel)
node_map = np.zeros((H, W), dtype=np.uint8)
node_map[rows_np, cols_np] = 1

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

# Left: node distribution with split lines
ax = axes[0]
ax.imshow(node_map, cmap='Greens', interpolation='nearest', aspect='equal')

ax.axhline(train_end, color='#2980B9', lw=2, linestyle='--',
           label=f'Train/Val split (row {train_end})')
ax.axhline(val_end,   color='#E67E22', lw=2, linestyle='--',
           label=f'Val/Test split  (row {val_end})')

# Shade the three regions
ax.axhspan(0,         train_end, alpha=0.04, color='#2ECC71', label='Train region')
ax.axhspan(train_end, val_end,   alpha=0.10, color='#F39C12', label='Val region')
ax.axhspan(val_end,   H,         alpha=0.04, color='#E74C3C', label='Test region')

ax.set_title(
    f'Subsampled node distribution — {graph.num_nodes:,} nodes\n'
    f'Geographic block split (north→south)',
    fontsize=11
)
ax.set_xlabel('Column (pixel)')
ax.set_ylabel('Row (pixel)')
ax.legend(loc='lower right', fontsize=8)
ax.set_axis_off()

# Right: burn probability spatial map (y_raw)
ax2 = axes[1]
bp_map = np.full((H, W), np.nan)
bp_map[rows_np, cols_np] = graph.y_raw.numpy().ravel()

im = ax2.imshow(bp_map, cmap='YlOrRd', vmin=0, vmax=0.12,
                interpolation='nearest', aspect='equal')
plt.colorbar(im, ax=ax2, label='Burn Probability', fraction=0.03, pad=0.02)

ax2.axhline(train_end, color='#2980B9', lw=1.5, linestyle='--')
ax2.axhline(val_end,   color='#E67E22', lw=1.5, linestyle='--')

ax2.set_title(
    f'Burn Probability (original scale) on subsampled nodes\n'
    f'min={float(graph.y_raw.min()):.5f}  '
    f'max={float(graph.y_raw.max()):.4f}  '
    f'mean={float(graph.y_raw.mean()):.5f}',
    fontsize=11
)
ax2.set_axis_off()

plt.tight_layout()
plt.savefig(FIG_DIR / 'p3_node_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: reports/figures/p3_node_distribution.png')